In [1]:
import os 
from glob import glob 

import numpy as np

import matplotlib.pyplot as plt

import xarray as xr
import pandas as pd
import dask

from sobol_sa import calculate_delta_full

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = "Arial"
plt.rcParams["font.size"] = 12
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["axes.linewidth"] = 1.50

from matplotlib.colors import LinearSegmentedColormap, ListedColormap

cm_data = np.loadtxt("../src/utils/colormaps/batlow.txt")[::-1]
sc_cmap = LinearSegmentedColormap.from_list("cmap", cm_data, N=10)

from utils.global_paths import project_data_path, project_code_path, loca_path
from utils.constants import obs_names, location_coords

In [2]:
############
### Dask ###
############
from dask_jobqueue import SLURMCluster

cluster = SLURMCluster(
    account="pches_cr_default",
    queue='basic',
    cores=1,
    memory="10GiB",
    walltime="01:00:00"
)
cluster.scale(jobs=20)  # ask for jobs

from dask.distributed import Client, get_worker, worker_client
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.6.10.89:45943,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


## Preliminaries

In [3]:
# Info 
subset_name = 'eCONUS'

# Metrics
soil_metrics_raw = ['mean', '5dmin', '5dmax']
soil_metrics_change = ['mean-change', '5dmin-change', '5dmax-change']
soil_metrics_days = ['days_above_q95', 'days_below_q05', 'days_above_q99', 'days_below_q01']

# Time slices to analyze
time_slices = [[2030, 2039], [2050,2059], [2080,2089]]

# SSPs
ssps = ['ssp245', 'ssp370']

In [4]:
# Models 
models = os.listdir(f"{loca_path}/")
models.remove('training_data')
models.remove('scripts')
models.remove('README.md')

loca_all = {}

# Loop through models
for model in models:
    loca_all[model] = {}
    # Loop through members
    members = os.listdir(f"{loca_path}/{model}/0p0625deg/")
    for member in members:
        # Append SSPs
        ssps = os.listdir(f"{loca_path}/{model}/0p0625deg/{member}/")
        loca_all[model][member] = ssps

# Matches website (https://loca.ucsd.edu/loca-version-2-for-north-america-ca-jan-2023/) as of Jan 2023
print(f"# models: {len(models)}")
print(f"# model/expts: {np.sum([len(np.unique([item for row in [loca_all[model][member] for member in loca_all[model].keys()] for item in row])) for model in models])}")
print(f"# model/expts/ens: {np.sum([len(loca_all[model][ssp]) for model in models for ssp in loca_all[model]])}")
print(f"# model/expts/ens (not including historical): {np.sum([len([ssp for ssp in loca_all[model][member] if ssp != 'historical']) for model in models for member in loca_all[model]])}")

# models: 27
# model/expts: 99
# model/expts/ens: 329
# model/expts/ens (not including historical): 221


In [5]:
# Read all soil moisture metrics
def read_all(subset_name, soil_metric, ssps):
    # For all
    ds_all = []

    # Loop through models
    for model in models:
        # Take first member only
        for member in list(loca_all[model].keys())[:1]:
            # Loop through SSPs
            for ssp in loca_all[model][member]:
                if ssp in ssps:
                    projection_id = f"{model}_{member}_{ssp}"
                    ds_proj = []
                    # Loop through obs
                    for obs_name in obs_names:
                        # Concat along loss metrics
                        ds = xr.open_mfdataset(f"{project_data_path}/projections/{subset_name}/metrics/{soil_metric}/{projection_id}_{obs_name}_*.nc",
                                                combine="nested", concat_dim = "loss_metric")
                        # Append
                        ds_proj.append(ds)
                    # Concat along obs
                    ds_proj = xr.concat(ds_proj, dim="obs_name")
                    ds_all.append(ds_proj)
                    
    # Concat along climate
    ds_all = xr.concat(ds_all, dim="projection_id")

    # Fix time dim
    ds_all['time'] = ds_all['time'].dt.year

    return ds_all

# ANOVA

In [40]:
def store_gridpoint(file_name, location_coords):
    # Read file
    ds = xr.open_dataset(file_name)
    metric = file_name.split('/')[-1][:-3]
    for location in list(location_coords.keys())[:1]:
        if not os.path.exists(f"{project_data_path}/projections/eCONUS/metrics_gridpoints/{location}_{metric}.csv"):
            # Subset location
            lat = location_coords[location][0]
            lon = location_coords[location][1]
            df = ds.sel(lat=lat, lon=lon, method='nearest').to_dataframe().reset_index()
            df = df.reset_index().drop(columns = ['index','lat','lon'])
            # Store
            df.to_csv(f"{project_data_path}/projections/eCONUS/metrics_gridpoints/{location}_{metric}.csv", index=False)
        else:
            df = pd.read_csv(f"{project_data_path}/projections/eCONUS/metrics_gridpoints/{location}_{metric}.csv")
            if len(df['loss_metric'].unique()) != 13:
                print(metric)

In [42]:
# Store results for R: changes
files = glob(f"{project_data_path}/projections/eCONUS/metrics/*.nc")

for file_name in files:
    store_gridpoint(file_name, location_coords)

5dmax_trend
5dmin_trend
days_above_q95_trend
days_above_q99_trend
days_below_q01_trend
days_below_q05_trend
mean_trend


In [6]:
# Store results for R: changes
files = glob(f"{project_data_path}/projections/eCONUS/metrics_combined/*.nc")

# delayed = []
# for file_name in files:
#     out = dask.delayed(store_gridpoint)(file_name, location_coords)
#     delayed.append(out)

# _ = dask.compute(*delayed)

In [13]:
# Store results for R: trends
files = glob(f"{project_data_path}/projections/eCONUS/metrics/*.nc")

# delayed = []
# for file_name in files:
#     out = dask.delayed(store_gridpoint)(file_name, location_coords)
#     delayed.append(out)

# _ = dask.compute(*delayed)

# Delta SA

## Changes from historical

### Functions

In [6]:
# Reads all .nc projections and stores as single netcdf
def store_combined(subset_name, soil_metric, time_slice):
    # Save path
    save_path = f'{project_data_path}/projections/{subset_name}/metrics_combined/{soil_metric}_{time_slice[0]}-{time_slice[1]}.nc'
    
    # Check if done
    if os.path.exists(save_path):
        print(f'{soil_metric} {time_slice} already done!')
    else:
        # Read all (all SSPs)
        ssps = ['ssp245', 'ssp370']
        ds_all = read_all(subset_name, soil_metric, ssps)
        
        # Storing dimensions as strings can cause issues - save as objects instead
        object_dimensions = ['loss_metric', 'projection_id', 'obs_name', 'ssp', 'member', 'model', 'soil_id']
        ds_all = ds_all.assign_coords({dimension_name: ds_all[dimension_name].astype(object) for dimension_name in object_dimensions})

        # Store
        ds_all.sel(time=slice(time_slice[0], time_slice[1])).to_netcdf(save_path)
        print(f'{soil_metric} {time_slice} done')

In [7]:
# Function to load the xarray dataset within a worker
def load_xarray_once(subset_name, soil_metric, time_slice):
    # Load worker
    worker = get_worker()
    
    # File name
    data_name = f'{soil_metric}_{time_slice[0]}-{time_slice[1]}'
    file_path = f'{project_data_path}/projections/{subset_name}/metrics_combined/{data_name}.nc'

    # Cache dataset in worker memory
    if data_name not in worker.data:
        worker.data[data_name] = xr.load_dataset(file_path)

    return worker.data[data_name]

In [8]:
# Delayed function to select grid point and perform SA
@dask.delayed
def get_delta(subset_name, soil_metric, time_slice, loc, sa_factors):
    # Load the dataset once per worker
    worker = get_worker()
    ds = load_xarray_once(subset_name, soil_metric, time_slice)

    # Get location as df
    lat, lon = loc
    df_loc = ds.sel(lat=lat, lon=lon).to_dataframe().reset_index()

    # If needed
    df_loc['soil_id'] = df_loc['obs_name'] + "_" + df_loc['loss_metric']
    
    # Shuffle for good measure
    df = df_loc.sample(frac=1)
    
    # Skip if all NaN
    if df.isnull().values.any():
        return None
    
    # Problem defn
    n_factors = len(sa_factors)
    problem = {
        'num_vars': n_factors,
        'names': sa_factors,
    }
    
    # Perform SA
    X = df[sa_factors].to_numpy()
    Y = df[soil_metric].to_numpy()

    Si = delta.analyze(problem, X, Y, num_resamples=2).to_df()
    Si['lat'] = lat
    Si['lon'] = lon
    Si = Si.reset_index().pivot(index=['lat','lon'], columns='index', values='delta')
    
    return Si

In [9]:
# Perform gridpoint-level delta SA on saved parquet file
def perform_delta_sa(subset_name, soil_metric, time_slice, sa_factors, save_name):
    # Check if done
    save_path = f'{project_data_path}/projections/{subset_name}/sa_results/{soil_metric}_{time_slice[0]}-{time_slice[1]}_delta-sa_{save_name}.nc'
    if os.path.exists(save_path):
        print(f'{soil_metric} {time_slice} already done!')
        return None
        
    # Get non-NaN locs
    locs = np.load(f"{project_code_path}/code/utils/grids/{subset_name}_non_nans.npy", allow_pickle=True)

    # Loop over all with dask.delayed
    delayed = [get_delta(subset_name, soil_metric, time_slice, loc, sa_factors) for loc in locs]
    
    # Compute
    delayed_out = dask.compute(*delayed)

    # Pandas dataframe 
    df = pd.concat(delayed_out)
    df = df.rename_axis(None, axis=1)

    # Create a complete grid of lat, lon values
    all_lats = np.load(f"{project_code_path}/code/utils/grids/{subset_name}_lat.npy")
    all_lons = np.load(f"{project_code_path}/code/utils/grids/{subset_name}_lon.npy")

    lon, lat = np.meshgrid(all_lons, all_lats)
    lon_lat_index = pd.MultiIndex.from_arrays([lat.flatten(), lon.flatten()], names=['lat', 'lon'])

    # Reindex to include all lat, lon combinations, filling missing ones with NaN
    df_reindexed = df.reindex(lon_lat_index)

    # Convert the reindexed DataFrame to an xarray Dataset
    ds = xr.Dataset.from_dataframe(df_reindexed)

    # Store
    ds.to_netcdf(save_path)
    print(f'{soil_metric} {time_slice} done')

In [10]:
# Store non-nan coords for subset if not done already
file_path = f"{project_code_path}/code/utils/grids/{subset_name}_non_nans.npy"

if not os.path.exists(file_path):
    # Read all 
    ds_all = read_all(subset_name, soil_metrics[0], ssps)

    # Get non-nans
    ds_all_stacked = ds_all.stack(loc=['lat','lon']).isel(time=10, projection_id=10, obs_name=0, loss_metric=0)[soil_metrics[0]]
    locs = ds_all_stacked[ds_all_stacked.notnull().compute()]['loc'].to_numpy()

    # Save
    np.save(file_path, locs)

### Calculations

In [11]:
# Store combined netCFDs
for soil_metric in soil_metrics_change + soil_metrics_days:
    for time_slice in time_slices:
        store_combined(subset_name, soil_metric, time_slice)

mean-change [2030, 2039] already done!
mean-change [2050, 2059] already done!
mean-change [2080, 2089] already done!
5dmin-change [2030, 2039] already done!
5dmin-change [2050, 2059] already done!
5dmin-change [2080, 2089] already done!
5dmax-change [2030, 2039] already done!
5dmax-change [2050, 2059] already done!
5dmax-change [2080, 2089] already done!
days_above_q95 [2030, 2039] already done!
days_above_q95 [2050, 2059] already done!
days_above_q95 [2080, 2089] already done!
days_below_q05 [2030, 2039] already done!
days_below_q05 [2050, 2059] already done!
days_below_q05 [2080, 2089] already done!
days_above_q99 [2030, 2039] done
days_above_q99 [2050, 2059] done
days_above_q99 [2080, 2089] done
days_below_q01 [2030, 2039] done
days_below_q01 [2050, 2059] done
days_below_q01 [2080, 2089] done


In [11]:
%%time
# Perform SA
for soil_metric in soil_metrics_change + soil_metrics_days:
    for time_slice in time_slices:
        # All
        sa_factors = ['ssp', 'model', 'time', 'obs_name', 'loss_metric']
        save_name = 'all'
        perform_delta_sa(subset_name, soil_metric, time_slice, sa_factors, save_name)
    
        # Soil grouped
        sa_factors = ['ssp', 'model', 'time', 'soil_id']
        save_name = 'soil_grouped'
        perform_delta_sa(subset_name, soil_metric, time_slice, sa_factors, save_name)

mean-change [2030, 2039] already done!
mean-change [2030, 2039] already done!
mean-change [2050, 2059] already done!
mean-change [2050, 2059] already done!
mean-change [2080, 2089] already done!
mean-change [2080, 2089] already done!
5dmin-change [2030, 2039] already done!
5dmin-change [2030, 2039] already done!
5dmin-change [2050, 2059] already done!
5dmin-change [2050, 2059] already done!
5dmin-change [2080, 2089] already done!
5dmin-change [2080, 2089] already done!
5dmax-change [2030, 2039] already done!
5dmax-change [2030, 2039] already done!
5dmax-change [2050, 2059] already done!
5dmax-change [2050, 2059] already done!
5dmax-change [2080, 2089] already done!
5dmax-change [2080, 2089] already done!
days_above_q95 [2030, 2039] already done!
days_above_q95 [2030, 2039] already done!
days_above_q95 [2050, 2059] already done!
days_above_q95 [2050, 2059] already done!
days_above_q95 [2080, 2089] already done!
days_above_q95 [2080, 2089] already done!
days_below_q05 [2030, 2039] alread

## Trends

### Calculation

In [6]:
# Linear regression function
def linear_regression(X, y):
    if np.isfinite(y).all() == False:
        return np.array([np.nan, np.nan])
    else:
        return np.polyfit(X, y, 1)

In [7]:
%%time
# SSPs
ssps = ['ssp245', 'ssp370']

# Loop through metrics
for soil_metric in ['mean', '5dmin', '5dmax', 'days_above_q95', 'days_below_q05', 'days_above_q99', 'days_below_q01']:
    # Check if done
    save_path = f'{project_data_path}/projections/eCONUS/metrics_combined/{soil_metric}_trend.nc'
    if not os.path.exists(save_path):
        # Real all
        ds = read_all(subset_name, soil_metric, ssps)

        # Linear trend
        result = xr.apply_ufunc(
            linear_regression,
            ds['time'],  # input x data
            ds[soil_metric],  # input y data
            input_core_dims=[['time'], ['time']],  # specify core dimensions for inputs
            output_core_dims=[['coef']],  # specify core dimensions for output
            vectorize=True,  # apply function element-wise
            dask='parallelized',  # enable parallelization with dask
            output_dtypes=[float],  # specify output data type
            dask_gufunc_kwargs={"output_sizes": {"coef": 2}},
        ).compute()
    
        # Store
        ds_result = xr.Dataset({'result': result})
        ds_result['coef'] = ['trend', 'intcp']

        encoding = {}
        for name, da in ds_result.variables.items():
            if str(da.dtype).startswith('<U'):
                encoding[name] = {'dtype': '<U100'}
            
        ds_result.to_netcdf(save_path, encoding=encoding)
        
    else:
        print(f"{soil_metric} already done!")

mean already done!
5dmin already done!
5dmax already done!
days_above_q95 already done!
days_below_q05 already done!
days_above_q99 already done!
days_below_q01 already done!
CPU times: user 838 μs, sys: 716 μs, total: 1.55 ms
Wall time: 7.21 ms


### SA

In [8]:
# Read specified lat/lon and perfrom delta SA
def get_delta(ds, lat, lon, sa_factors):
    # Select
    df_loc = ds.sel(coef='trend').sel(lat=lat, lon=lon).to_dataframe().reset_index()

    # If needed
    df_loc['soil_id'] =  df_loc['obs_name'] + "_" + df_loc['loss_metric']

    # Shuffle for good measure
    df = df_loc.sample(frac=1)

    # Skip if all NaN
    if df.isnull().values.any():
        return None

    # Problem defn
    n_factors = len(sa_factors)
    problem = {
        'num_vars': n_factors,
        'names': sa_factors,
    }

    # Perform SA
    X = df[sa_factors].to_numpy()
    Y = df['result'].to_numpy()
    
    # Si = delta.analyze(problem, X, Y, num_resamples=2, method='delta').to_df()
    
    indices = calculate_delta_full(Y, X)
    Si = pd.DataFrame(data=indices, index=sa_factors, columns=['delta'])
    Si['lat'] = lat
    Si['lon'] = lon

    Si = Si.reset_index().pivot(index=['lat','lon'], columns='index', values='delta')
    
    return Si

In [9]:
%%time

sa_grouping = {'soil_grouped': ['soil_id', 'ssp', 'model'],
               'all': ['ssp', 'model', 'obs_name', 'loss_metric']}

# Loop through SA grouping 
for save_name in sa_grouping.keys():
    sa_factors = sa_grouping[save_name]
    
    # Loop through metrics
    for soil_metric in ['mean', '5dmin', '5dmax', 'days_above_q95', 'days_below_q05', 'days_above_q99', 'days_below_q01']:
        save_path = f'{project_data_path}/projections/{subset_name}/sa_results/{soil_metric}_trends_delta-sa_{save_name}.nc'
        if os.path.exists(save_path):
            print(f'{soil_metric} already done!')

        else:
            # Read trends
            ds = xr.open_dataset(f'{project_data_path}/projections/{subset_name}/metrics_combined/{soil_metric}_trend.nc')
        
            # Get non-NaN locs
            locs = np.load(f"{project_code_path}/src/utils/grids/{subset_name}_non_nans.npy", allow_pickle=True)
        
            # Loop over all with dask.delayed
            delayed = []
            for loc in locs:
                lat, lon = loc
                df_tmp = dask.delayed(get_delta)(ds, lat, lon, sa_factors)
                delayed.append(df_tmp)
        
            # Compute
            delayed_out = dask.compute(*delayed)
        
            # Pandas dataframe 
            df = pd.concat(delayed_out)
            df = df.rename_axis(None, axis=1)
            
            # Create a complete grid of lat, lon values
            all_lats = np.load(f"{project_code_path}/src/utils/grids/{subset_name}_lat.npy")
            all_lons = np.load(f"{project_code_path}/src/utils/grids/{subset_name}_lon.npy")
            
            lon, lat = np.meshgrid(all_lons, all_lats)
            lon_lat_index = pd.MultiIndex.from_arrays([lat.flatten(), lon.flatten()], names=['lat', 'lon'])
            
            # Reindex to include all lat, lon combinations, filling missing ones with NaN
            df_reindexed = df.reindex(lon_lat_index)
            
            # Convert the reindexed DataFrame to an xarray Dataset
            ds_out = xr.Dataset.from_dataframe(df_reindexed)
        
            # Store
            ds_out.to_netcdf(save_path)
            print(soil_metric)

mean already done!
5dmin already done!
5dmax
days_above_q95
days_below_q05
days_above_q99
days_below_q01
mean
5dmin
5dmax
days_above_q95
days_below_q05
days_above_q99
days_below_q01
CPU times: user 33min 45s, sys: 51.2 s, total: 34min 37s
Wall time: 38min 50s
